## SVM
SVM tries to find a decision boundary with the maximum margin that separates classes. 

Math formulation
$$y_i(w^Tx_i + b) \geq 1$$

SVM normalizes the decision function so that support vectors lie on $w^Tx + b = \pm 1$, and the perpendicular distance between these two parallel hyperplanes is margin size: 
$$\frac{2}{\lVert w \rVert}$$

So maximizing margin = minimizing:
$$min_{x} \frac{1}{2} \lVert w\rVert^2$$

Data is rarely perfectly separable, so we introduce slack:
$$min_{x} \frac{1}{2} \lVert w\rVert^2 + C\sum \varepsilon_i$$


### Hinge loss

SVM minimizes:
$$max(0, 1-yf(x))$$

- correctly classified but too close -> still penalized
- correctly classified and far -> no penalty


### Kernel trick

SVM can become nonlinear without explicitly creating features 
We compute 
$$K(x_i, x_j) = \phi(x_i)^T\phi(x_j)$$

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

class LinearSVM(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1) #input, output

    def forward(self, x):
        return self.linear(x).squeeze(1)

### Hinge Loss
Used for binary classification with labels $y \in \{-1, +1\}$
$$L = max(0, 1-y\cdot f(x))$$

Intuition:
we want $$y\cdot f(x) \geq 1$$
- correct sign
- far enough from decision boundary

In [5]:
## Hinge loss + regularization 
def svm_loss(model, x, y, C):
    """
    x: shape [N, D]
    y: shape [N], with values in {-1, +1}
    """

    score = model(x) 
    hinge = torch.clamp(1 - y*score, min=0).mean()

    w = model.linear.weight
    reg = 0.5 * torch.sum(w*w)
    return reg + C * hinge

    

In [6]:
# Example data
torch.manual_seed(0)
N, D = 200, 2
x_pos = torch.randn(N//2, D) + 2.0 
x_neg = torch.randn(N//2, D) - 2.0

X = torch.cat([x_pos, x_neg], dim=0)
y = torch.cat([
    torch.ones(N//2),
    -torch.ones(N//2)
    ], dim=0)


In [8]:
##Train
model = LinearSVM(input_dim=D)
optimizer = optim.SGD(model.parameters(), lr=0.01)

num_epochs=200
C = 1.0
for epoch in range(num_epochs):
    optimizer.zero_grad()
    loss = svm_loss(model, X, y, C=C)
    loss.backward()
    optimizer.step()


    if epoch % 100 == 0:
        with torch.no_grad():
            scores = model(X)
            preds = torch.sign(scores)
            acc = (preds == y).float().mean().item()
        print(f"epoch={epoch:4d} loss={loss.item():.4f} acc={acc:.4f}")

# ----------------------------
# Inference
# ----------------------------
with torch.no_grad():
    scores = model(X)
    preds = torch.sign(scores)   # -1 or +1
    

epoch=   0 loss=1.2330 acc=0.4700
epoch= 100 loss=0.2170 acc=0.9950


### Intuition
```
loss.backward() # compute gradient
optimizer.step() # update parameters 
```

When you run 
```
scores = model(x)
loss = loss_fn(scores, y)
```
Pytorch builds a graph of operations like: `x -> linear -> score -> loss`. 
Each node knows:
- forward value
- how to compute gradient, stored in `param.grad`

when you call `loss.backward()`
PyTorch:
- Starts from loss
- Applies chain rule backward
- Computes gradients for all tensors with `requires_grad=True`

Note that gradients accumulate if u call `backward()` twice. So you need to clear them before computing new ones: 
```
optimizer.zero_grad()
```


`optimizer.step()` updates the weights using the gradients

for simple SGD:

$$w = w - \eta \cdot \nabla_w loss$$

### Full Training Loop
```
for batch in data:
    optimizer.zero_grad()
    scores = model(x)
    loss = loss_fn(scores)
    loss.backward()
    optimizer.step()
```